In [1]:
import SuperconductingDetector as sd
import numpy as np

## Defining superconductors

In [2]:
NbTiN_properties = dict(
    Tc  = 15,     # K
    rho = 217e-8, # Ohm m
    d   = 200e-9, # m
)
NbTiN = sd.CPW.material("NbTiN", **NbTiN_properties)

bTa_properties = dict(
    Tc  = 0.6,    # K
    rho = 49e-8,  # Ohm m
    d   = 100e-9, # m
)
bTa = sd.CPW.material("\u03B2-Ta", **bTa_properties)

In [3]:
# Plotting complex conductivity (using Mattis-Bardeen equations)
f = np.logspace(np.log10(bTa.fc*0.1), np.log10(bTa.fc*10), 100) # Hz
T = bTa.Tc/10 # K

sd.plotly([f/bTa.fc]*2, bTa.sigma12(f, T), IDs=["\u03c3\u2081","\u03c3\u2082"],
          title_text=bTa.name, labels=dict(x="f\:/\:f_c",x_unit="",y="\sigma_i\:/\:\sigma_N",y_unit=""),
          vlines=[1], hlines=[1], template="base+light+xlog+ylog")

C:\Users\matth\Anaconda\MSc AP\PythonMEP\SuperconductingDetector\CPW.py:164: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  sigma_2[i]  = quad(int_2, -Delta,     Delta, args=(hf), points=[Delta-hf, -Delta], **intkwargs)[0]


## Defining CPWs

In [4]:
NbTiN_CPW_properties = dict(
    S = 20e-6,  # m
    W = 8e-6, # m
    material = NbTiN,
)
NbTiN_CPW = sd.CPW.create(**NbTiN_CPW_properties)

hybrid_CPW_properties = dict(
    S = 3e-6, # m
    W = 2e-6, # m
    material = NbTiN,
)
hybrid_CPW = sd.CPW.create(**hybrid_CPW_properties)
hybrid_CPW.make_hybrid(bTa)

C:\Users\matth\Anaconda\MSc AP\PythonMEP\SuperconductingDetector\CPW.py:419: UserWarning: Approximation geometric contributions is not valid using these values for S, W, and d.
  warnings.warn("Approximation geometric contributions is not valid using these values for S, W, and d.")


In [5]:
# Plotting schematic cross section
hybrid_CPW.draw()

In [6]:
# Plotting characteristic impedance of the hybrid CPW
f = np.logspace(9,13,300) # Hz
T = bTa.Tc/10 # K
Z0 = hybrid_CPW.Z0(f,T)

sd.plotly([f]*2, [np.real(Z0), np.imag(Z0)], IDs=["Re{Z\u2080}","Im{Z\u2080}"], 
          labels=dict(x="f",y="Z_0",x_unit="Hz",y_unit="\u2126"), title_text=hybrid_CPW.name, 
          vlines=[hybrid_CPW.mat_c.fc, hybrid_CPW.mat_g.fc], hlines=[0], show_annotations=True, 
          template="base+light+xlog", legend="middle left")

C:\Users\matth\Anaconda\MSc AP\PythonMEP\SuperconductingDetector\CPW.py:164: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  sigma_2[i]  = quad(int_2, -Delta,     Delta, args=(hf), points=[Delta-hf, -Delta], **intkwargs)[0]
C:\Users\matth\Anaconda\MSc AP\PythonMEP\SuperconductingDetector\CPW.py:160: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  sigma_1[i] =      quad(int_1a, Delta,   intlim1, args=(hf), points=Delta, **intkwargs)[0]


## Defining KID structures

In [7]:
KID_simple = sd.KID.simple(hybrid_CPW, T=bTa.Tc/10, mode="readout")

In [8]:
f0 = 3e9 # Hz
Qc = 1e5
S21_fit = KID_simple.S21_fit(f0, Qc, show_plot=True)
for key in S21_fit:
    print(f"{key} = {S21_fit[key]:.3g}")

f0 = 3e+09
S21_min = 0.00115
FWHM = 2.99e+04
Ql = 1e+05
Qi = 8.72e+07
Qc = 1.01e+05
r = 0.499


In [9]:
KID_hybrid = sd.KID.hybrid(hybrid_CPW, NbTiN_CPW, l_hybrid=300e-6, mode="readout")

In [10]:
f0 = 3e9 # Hz
Qc = 1e4

f = f0 + np.linspace(-0.01, 0, 5000)*1e9 # Hz

# Adding antenna impedance at the load
Z_antenna = 10j*np.ones(np.shape(f)) # Example impedance
KID_hybrid.set_Z_L(f, Z_antenna)

S21 = KID_hybrid.S21(f, f0, Qc)

print("Kinetic inductance fraction = {:.3f}".format(KID_hybrid.alpha_k(f0, Qc)))

sd.plotly(np.real(S21), np.imag(S21), size=(400,400), xrange=[0,1], yrange=[-0.5,0.5],
          labels=dict(x="Re\{S_{21}\}",x_unit="\Omega",y="Im\{S_{21}\}",y_unit="\Omega"))

Kinetic inductance fraction = 0.711


## Twin-slot antenna

In [11]:
dims = dict(L_ant=1200, W_ant=686, W_1=168, W_2=48, d_ant=72) # um
antenna = sd.antenna.create(**dims)

antenna.draw(size=600, TL=KID_hybrid)

## Extended hemispherical lens

In [12]:
D = 8.995    # mm
theta0 = 50  # degrees
ext = 0.37   # *R_sphere
lens = sd.lens.create(D, theta0, ext)

lens.draw()